In [3]:
import getdist.plots as gdplt
from cobaya import load_samples
import matplotlib.pyplot as plt
import numpy as np
import camb
import cosmoprimo
import os
# from scipy.stats import linregress
# from scipy.optimize import curve_fit

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'

plt.rc('xtick', labelsize=12) 
plt.rc('ytick', labelsize=12)
plt.rc('font', size=14)
plt.rc('axes', labelsize=16)

# looking at distances values on degeneracy line

#### for different lines

In [17]:
def get_DESI_errors():
    dict_distance, z_DESI, path = get_DESI_data()

    cov_mat = np.loadtxt(path.replace('mean', 'cov'))

    errors = {}
    for i in range(len(cov_mat[0, :])):
        z = z_DESI[i//2]
        if i == 0:
            _, _, DVfid = get_DESI_fid(z)
            a_iso_err = np.sqrt(cov_mat[i][i]) / DVfid
            errors[z] = {'d_iso': a_iso_err}
        elif i%2 == 0:
            cov = [[cov_mat[i-1, i-1], cov_mat[i-1, i]], [cov_mat[i, i-1], cov_mat[i, i]]]
            sig_x = np.sqrt(cov[0][0])
            sig_y = np.sqrt(cov[1][1])

            DMfid, DHfid, DVfid = get_DESI_fid(z)
            if i == len(cov_mat[0, :])-1:
                d_a_perp = sig_y / DMfid
                d_a_par = sig_x / DHfid                
            else :
                d_a_perp = sig_x / DMfid
                d_a_par = sig_y / DHfid

            DM = dict_distance[z]['DM_over_rs']
            DH = dict_distance[z]['DH_over_rs']
            DV = (z*DM**2*DH)**(1/3)

            a_perp = DM / DMfid
            a_par = DH / DHfid
            a_iso = DV / DVfid
            a_AP = a_par / a_perp

            d_a_AP = a_AP * np.sqrt((d_a_par / a_par)**2 + (d_a_perp / a_perp)**2)
            d_a_iso = a_iso * np.sqrt((2/3)**3 * (d_a_perp / a_perp)**2 + (1/3)**2 * (d_a_par / a_par)**2)

            errors[z] = {'d_perp':  d_a_perp,
                         'd_par':   d_a_par,
                         'd_AP':    d_a_AP,
                         'd_iso':   d_a_iso
                         }
    return errors, z_DESI

def get_DESI_data():
    z_DESI = []
    path = '/home/adrien/PDM/code/desi_bao_dr2/desi_gaussian_bao_ALL_GCcomb_mean.txt'
    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.split()
            z_DESI.append(float(parts[0]))

    z_DESI = set(z_DESI)
    z_DESI = list(z_DESI)
    z_DESI.sort()
    dict_distance = {}
    for z in z_DESI:
        dict_distance[z] = {}

    with open(path, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.split()
            z = float(parts[0])
            for key in dict_distance.keys():
                if key == z:
                    dict_distance[key][parts[2]] = float(parts[1])
    
    return dict_distance, z_DESI, path

def get_DESI_fid(redshift):
    DESI = cosmoprimo.fiducial.DESI(engine='class')
    DESI_bkg = DESI.get_background()
    DESI_thermo = DESI.get_thermodynamics()
    rdrag_fid = DESI_thermo.rs_drag
    DM_fid = DESI_bkg.comoving_angular_distance(redshift)
    DH_fid = 1 / DESI_bkg.efunc(redshift) * 2997.92458 # c/H(z) in Mpc, c=299792 km/s, H0 in km/s/Mpc
    DV_fid = (redshift * DM_fid**2 * DH_fid)**(1/3)
    DMover_rd_fid = DM_fid / rdrag_fid
    DHover_rd_fid = DH_fid / rdrag_fid
    DVover_rd_fid = DV_fid / rdrag_fid

    return DMover_rd_fid, DHover_rd_fid, DVover_rd_fid

In [10]:
cov_DESI2nP1 = np.loadtxt('/home/adrien/PDM/code/PDM2026_wsl/cov_mat/DESI2nP=1/cov_DM_DH.txt')

In [14]:
def extract_error_bars(cov_mat, DESI_style=False):
    n = len(cov_mat[0, :])
    err_DM = []
    err_DH = []

    if DESI_style:
        print("need to be coded")    
    else:
        for i in range(n):
            if i%2 == 0:
                sig_x = np.sqrt(cov_mat[i][i])
                sig_y = np.sqrt(cov_mat[i+1][i+1])
                err_DM.append(sig_x)
                err_DH.append(sig_y)
    
    return err_DM, err_DH

In [ ]:
err_DM, err_DH = extract_error_bars(cov_DESI2nP1)

[np.float64(0.10899568798810345), np.float64(0.11039121387139468), np.float64(0.10025424978523355), np.float64(0.10694844926411977), np.float64(0.11746494711189377), np.float64(0.15041042849483544), np.float64(0.17962616234836173)]


In [20]:
z = np.linspace(0.01, 3, 100)

# DM, DH, DV = get_DESI_fid(z)

DESI_errors, z_DESI = get_DESI_errors()

plt.figure(figsize=(8, 6))
plt.errorbar(z_DESI, [1 for _ in z_DESI], yerr=DESI_errors[z_DESI[0]]['d_iso'], fmt='o', label='DM', color='blue')

ModuleNotFoundError: No module named 'pyclass'

In [ ]:
def plot_DM_DH_2panels(redshift, slope, offset, param_cosmo, ylim=None, yticks=None, text_loc='lower right', 
                                name='', path_save=None, xlabel=True, show_plots=True, DESI_errors=None,
                                envelope_settings=None, symmetric=False, AP_ISO = True):
    
    if AP_ISO:
        alph_AP   = []
        alph_ISO  = []
    else:
        alph_perp = []
        alph_par  = []

    w0 = [cosmo[0] for cosmo in param_cosmo]

    for i in range(len(w0)):
        a, b, c = alpha_perp_parallel(redshift, param_cosmo[i])
        if AP_ISO:
            alph_AP.append(b / a)
            alph_ISO.append(c)
        else:
            alph_perp.append(a)
            alph_par.append(b)

    cmap = cm.viridis
    w0min, w0max = min(w0), max(w0)

    if symmetric:
        if w0max < -0.8: w0max = -0.8
        if w0min > -1.2: w0min = -1.2
    else:
        if w0max < -0.5: w0max = -0.5
        if w0min > -1.1: w0min = -1.1

    norm = mcolors.Normalize(vmin=w0min, vmax=w0max)

    fig = plt.figure(figsize=(10, 3.5))
    gs = fig.add_gridspec(
            1, 2,
        wspace=0.32, hspace=0.3
    )

    ax00 = fig.add_subplot(gs[0, 0])
    ax01 = fig.add_subplot(gs[0, 1])

    axes = {'00': ax00, '01': ax01}
    if AP_ISO:
        data = {'00': alph_ISO, '01': alph_AP}
        ylabels = {
            '00': r'$\alpha_\mathrm{ISO}(z)$',
            '01': r'$\alpha_\mathrm{AP}(z)$'
            }        
    else:
        data = {'00': alph_perp, '01': alph_par}
        ylabels = {
            '00': r'$\alpha_\perp(z)$',
            '01': r'$\alpha_\parallel(z)$'
            }


        # if envelope_settings is not None:
        #     envelope_type = envelope_settings['type']
        #     if envelope_type == 'spline':
        #         a = envelope_settings['alpha']
        #         k = envelope_settings['k']
        #         smooth = envelope_settings['smooth']
        #         from scipy.interpolate import UnivariateSpline
        #         z_fine = np.linspace(0, 3, 200)
        #         spline = UnivariateSpline(z_DESI_uncert, a_perp_err, s=smooth, k=k)  # s contrôle le lissage
        #         ax10.fill_between(z_fine, 1 - spline(z_fine), 1 + spline(z_fine), color='k', alpha=a)
        
        # from scipy.interpolate import interp1d

        # z_fine = np.linspace(min(z_DESI_uncert), max(z_DESI_uncert), 200)
        # z_fine = np.linspace(0, 3, 200)
        # err_interp = interp1d(z_DESI_uncert, a_perp_err, kind='cubic', fill_value='extrapolate')
        # ax10.fill_between(z_fine, 1 - err_interp(z_fine), 1 + err_interp(z_fine), color='k', alpha=0.2)
    
    if DESI_errors is not None:
        uncert, z_uncert = get_DESI_errors()
        a_perp_err = []
        a_par_err = []
        a_iso_err = []
        a_AP_err = []
        for idx, (_, value) in enumerate(uncert.items()):
            if idx == 0:
                a_iso_err.append(value['d_iso'])
            else:
                a_perp_err.append(value['d_perp'])
                a_par_err.append(value['d_par'])
                a_AP_err.append(value['d_AP'])
                a_iso_err.append(value['d_iso'])
        a_err = [a_iso_err, a_AP_err, a_perp_err, a_par_err]
    
    if AP_ISO:
        a_err = a_err[:2]
    else:
        a_err = a_err[2:]

    for idx, (key, ax) in enumerate(axes.items()):
        for i in range(len(w0)):
            ax.plot(redshift, data[key][i], color=cmap(norm(w0[i])))
    
        if DESI_errors is not None:
            if idx == 0 and AP_ISO:
                ax.errorbar(z_uncert, [1 for _ in z_uncert], yerr=a_err[idx], fmt='o', color='k', markersize=3)
            else:
                ax.errorbar(z_uncert[1:], [1 for _ in z_uncert[1:]], yerr=a_err[idx], fmt='o', color='k', markersize=3)

        ax.axhline(1, color='k', ls='--', lw=1)

        ax.tick_params(which='major', direction='in', length=4, width=1,   top=True, right=True)
        ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, right=True)
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())

        ax.set_ylabel(ylabels[key], fontsize=18)
        ax.set_xlim(0, 3)

        if ylim is not None:
            if 'all' in ylim:
                ax.set_ylim(ylim['all'])
            elif key in ylim:
                ax.set_ylim(ylim[key])

        if yticks is not None:
            if 'all' in yticks:
                ax.yaxis.set_major_locator(MultipleLocator(yticks['all']))
            elif key in yticks:
                ax.yaxis.set_major_locator(MultipleLocator(yticks[key]))

        if xlabel and key in ('00', '01'):
            ax.set_xlabel('Redshift', fontsize=16)

    leg_txt = rf'$w_a = {slope:.2f}(w_0 + {offset:.1f})$'
    if slope == 0.0: leg_txt = rf'$w_a = 0$'
    text = AnchoredText(
        leg_txt,
        loc=text_loc, frameon=True, prop=dict(size=14)
    )
    axes['00'].add_artist(text)

    # --- Colorbar à droite, taille ~1 subplot, centrée ---
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar_ax = fig.add_axes([0.92, 0.12, 0.015, 0.74])
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label(r'$w_0$', fontsize=16)
    cb.ax.tick_params(labelsize=11)

    if path_save is not None:
        slope_str  = f'{slope:.2f}'
        offset_str = f'{offset:.1f}'
        
        if not xlabel:
            folder = r'\no_xlabel'
            pth_folder =  path_save + folder
            os.makedirs(pth_folder, exist_ok=True)
            fname = folder + rf'\{name}_{slope_str}_{offset_str}' if name else rf'{slope_str}_{offset_str}'
        else:
            fname = f'{name}_{slope_str}_{offset_str}' if name else f'{slope_str}_{offset_str}'

        if path_save is None:
            path_save = rf'..\..\figures\degen_line\alpha_ratio\final\{fname}.png'
        else:
            path_save= path_save + f'\{fname}.png'
        fig.savefig(
            path_save,
            dpi=150, bbox_inches='tight'
        )

    if show_plots:
        plt.show()
    else:
        plt.close(fig)

    return fig, axes

In [5]:
def alpha_perp_parallel(z, cosmo_params):
    '''
    alpha_perp = (DM(z)/rd) / (DM_fid(z)/rd_fid)
    alpha_parallel = (DH(z)/rd) / (DH_fid(z)/rd_fid)
    fid --> the DESI fiducial cosmology
    '''

    w0, wa, Omega_m, hrdrag = cosmo_params

    DESI = cosmoprimo.fiducial.DESI(engine='camb')
    DESI_bkg = DESI.get_background()
    DESI_thermo = DESI.get_thermodynamics()
    rdrag_fid = DESI_thermo.rs_drag
    DM_fid = DESI_bkg.comoving_angular_distance(z)
    DH_fid = 1 / DESI_bkg.efunc(z) * 2997.92458 # c/H(z) in Mpc, c=299792 km/s, H0 in km/s/Mpc
    DV_fid = (z * DM_fid**2 * DH_fid)**(1/3)
    DMover_rd_fid = DM_fid / rdrag_fid
    DHover_rd_fid = DH_fid / rdrag_fid
    DVover_rd_fid = DV_fid / rdrag_fid

    custom_cosmo = cosmoprimo.Cosmology(w=w0, wa=wa, Omega_m=Omega_m)
    custom_bkg = custom_cosmo.get_background(engine='camb')
    DM_real = custom_bkg.comoving_angular_distance(z)
    DH_real = 1 / custom_bkg.efunc(z) * 2997.92458 # c/H(z) in Mpc, c=299792 km/s, H0 in km/s/Mpc
    DV_real = (z * DM_real**2 * DH_real)**(1/3)
    DMover_rd_real = DM_real / hrdrag
    DHover_rd_real = DH_real / hrdrag
    DVover_rd_real = DV_real / hrdrag
    
    return DMover_rd_real / DMover_rd_fid, DHover_rd_real / DHover_rd_fid, DVover_rd_real / DVover_rd_fid

In [6]:
def get_Om_hrdrag(w0, wa):
    '''
    For each (w0, wa) pair, uses CAMB to get Omega_m, h*rdrag, h
    by fixing theta_star=0.01041, omega_b=0.02223, omega_bc=0.14208
    Dark energy initialized first, H0 set to None.
    
    Accepts scalars or lists/arrays for w0, wa.
    Returns Omega_m, hrdrag, h (arrays if input is array)
    '''

    scalar_input = np.ndim(w0) == 0
    w0 = np.atleast_1d(w0)
    wa = np.atleast_1d(wa)

    Omega_m_list = []
    hrdrag_list = []
    h_list = []

    for w0_i, wa_i in zip(w0, wa):
        pars = camb.CAMBparams()
        
        # Initialize dark energy first
        pars.set_dark_energy(w=w0_i, wa=wa_i, dark_energy_model='ppf')
        
        pars.set_cosmology(
            thetastar=0.01041,
            ombh2=0.02223,
            omch2=0.14208-0.02223,
            H0=None
        )

        results = camb.get_background(pars)
        
        h = pars.h
        Omega_m = pars.omegam
        rdrag = results.get_derived_params()['rdrag']
        hrdrag = h * rdrag

        Omega_m_list.append(Omega_m)
        hrdrag_list.append(hrdrag)
        h_list.append(h)

    Omega_m_arr = np.array(Omega_m_list)
    hrdrag_arr = np.array(hrdrag_list)
    h_arr = np.array(h_list)

    if scalar_input:
        return Omega_m_arr[0], hrdrag_arr[0], h_arr[0]
    
    return Omega_m_arr, hrdrag_arr, h_arr

In [7]:
from matplotlib.offsetbox import AnchoredText
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
import matplotlib.cm as cm
import matplotlib.colors as mcolors

def plot_alpha_ratios(redshift, slope, offset, param_cosmo, ylim=None, yticks=None, text_loc='lower right', 
                      name='', path_save=None, xlabel=True, show_plots=True, DESI_errors=None,
                      envelope_settings=None, symmetric=False):
    
    alph_perp = []
    alph_par  = []
    alph_AP   = []
    alph_ISO  = []

    w0 = [cosmo[0] for cosmo in param_cosmo]

    for i in range(len(w0)):
        a, b, c = alpha_perp_parallel(redshift, param_cosmo[i])
        alph_perp.append(a)
        alph_par.append(b)
        alph_AP.append(b / a)
        alph_ISO.append(c)

    cmap = cm.viridis
    w0min, w0max = min(w0), max(w0)

    if symmetric:
        if w0max < -0.8: w0max = -0.8
        if w0min > -1.2: w0min = -1.2
    else:
        if w0max < -0.5: w0max = -0.5
        if w0min > -1.1: w0min = -1.1

    norm = mcolors.Normalize(vmin=w0min, vmax=w0max)

    fig = plt.figure(figsize=(10, 7))
    gs = fig.add_gridspec(
            2, 2,
        wspace=0.32, hspace=0.3
    )

    ax00 = fig.add_subplot(gs[0, 0])
    ax01 = fig.add_subplot(gs[0, 1])
    ax10 = fig.add_subplot(gs[1, 0])
    ax11 = fig.add_subplot(gs[1, 1])

    axes = {'00': ax00, '01': ax01, '10': ax10, '11': ax11}
    data = {'00': alph_ISO, '01': alph_AP, '10': alph_perp, '11': alph_par}
    ylabels = {
        '00': r'$\alpha_\mathrm{ISO}(z)$',
        '01': r'$\alpha_\mathrm{AP}(z)$',
        '10': r'$\alpha_\perp(z)$',
        '11': r'$\alpha_\parallel(z)$',
    }

        # if envelope_settings is not None:
        #     envelope_type = envelope_settings['type']
        #     if envelope_type == 'spline':
        #         a = envelope_settings['alpha']
        #         k = envelope_settings['k']
        #         smooth = envelope_settings['smooth']
        #         from scipy.interpolate import UnivariateSpline
        #         z_fine = np.linspace(0, 3, 200)
        #         spline = UnivariateSpline(z_DESI_uncert, a_perp_err, s=smooth, k=k)  # s contrôle le lissage
        #         ax10.fill_between(z_fine, 1 - spline(z_fine), 1 + spline(z_fine), color='k', alpha=a)
        
        # from scipy.interpolate import interp1d

        # z_fine = np.linspace(min(z_DESI_uncert), max(z_DESI_uncert), 200)
        # z_fine = np.linspace(0, 3, 200)
        # err_interp = interp1d(z_DESI_uncert, a_perp_err, kind='cubic', fill_value='extrapolate')
        # ax10.fill_between(z_fine, 1 - err_interp(z_fine), 1 + err_interp(z_fine), color='k', alpha=0.2)

    if DESI_errors is not None:
        uncert, z_uncert = get_DESI_errors()
        a_perp_err = []
        a_par_err = []
        a_iso_err = []
        a_AP_err = []
        for idx, (_, value) in enumerate(uncert.items()):
            if idx == 0:
                a_iso_err.append(value['d_iso'])
            else:
                a_perp_err.append(value['d_perp'])
                a_par_err.append(value['d_par'])
                a_AP_err.append(value['d_AP'])
                a_iso_err.append(value['d_iso'])
        a_err = [a_iso_err, a_AP_err, a_perp_err, a_par_err]

    for idx, (key, ax) in enumerate(axes.items()):
        for i in range(len(w0)):
            ax.plot(redshift, data[key][i], color=cmap(norm(w0[i])))
    
        if DESI_errors is not None:
            if idx == 0:
                ax.errorbar(z_uncert, [1 for _ in z_uncert], yerr=a_err[idx], fmt='o', color='k', markersize=3)
            else:
                ax.errorbar(z_uncert[1:], [1 for _ in z_uncert[1:]], yerr=a_err[idx], fmt='o', color='k', markersize=3)

        ax.axhline(1, color='k', ls='--', lw=1)

        ax.tick_params(which='major', direction='in', length=4, width=1,   top=True, right=True)
        ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, right=True)
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())

        ax.set_ylabel(ylabels[key], fontsize=18)
        ax.set_xlim(0, 3)

        if ylim is not None:
            if 'all' in ylim:
                ax.set_ylim(ylim['all'])
            elif key in ylim:
                ax.set_ylim(ylim[key])

        if yticks is not None:
            if 'all' in yticks:
                ax.yaxis.set_major_locator(MultipleLocator(yticks['all']))
            elif key in yticks:
                ax.yaxis.set_major_locator(MultipleLocator(yticks[key]))

        if xlabel and key in ('10', '11'):
            ax.set_xlabel('Redshift', fontsize=16)

    leg_txt = rf'$w_a = {slope:.2f}(w_0 + {offset:.1f})$'
    if slope == 0.0: leg_txt = rf'$w_a = 0$'
    text = AnchoredText(
        leg_txt,
        loc=text_loc, frameon=True, prop=dict(size=14)
    )
    axes['00'].add_artist(text)

    # --- Colorbar à droite, taille ~1 subplot, centrée ---
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar_ax = fig.add_axes([0.92, 0.25, 0.015, 0.5])
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label(r'$w_0$', fontsize=16)
    cb.ax.tick_params(labelsize=11)

    if path_save is not None:
        slope_str  = f'{slope:.2f}'
        offset_str = f'{offset:.1f}'
        
        if not xlabel:
            folder = r'\no_xlabel'
            pth_folder =  path_save + folder
            os.makedirs(pth_folder, exist_ok=True)
            fname = folder + rf'\{name}_{slope_str}_{offset_str}' if name else rf'{slope_str}_{offset_str}'
        else:
            fname = f'{name}_{slope_str}_{offset_str}' if name else f'{slope_str}_{offset_str}'

        if path_save is None:
            path_save = rf'..\..\figures\degen_line\alpha_ratio\final\{fname}.png'
        else:
            path_save= path_save + f'\{fname}.png'
        fig.savefig(
            path_save,
            dpi=150, bbox_inches='tight'
        )

    if show_plots:
        plt.show()
    else:
        plt.close(fig)

    return fig, axes

<string>:157: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:157: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<string>:157: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:157: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
/tmp/ipykernel_55576/1733381929.py:157: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
  path_save= path_save + f'\{fname}.png'


In [ ]:
def plot_alpha_ratios_2panels(redshift, slope, offset, param_cosmo, ylim=None, yticks=None, text_loc='lower right', 
                                name='', path_save=None, xlabel=True, show_plots=True, DESI_errors=None,
                                envelope_settings=None, symmetric=False, AP_ISO = True):
    
    if AP_ISO:
        alph_AP   = []
        alph_ISO  = []
    else:
        alph_perp = []
        alph_par  = []

    w0 = [cosmo[0] for cosmo in param_cosmo]

    for i in range(len(w0)):
        a, b, c = alpha_perp_parallel(redshift, param_cosmo[i])
        if AP_ISO:
            alph_AP.append(b / a)
            alph_ISO.append(c)
        else:
            alph_perp.append(a)
            alph_par.append(b)

    cmap = cm.viridis
    w0min, w0max = min(w0), max(w0)

    if symmetric:
        if w0max < -0.8: w0max = -0.8
        if w0min > -1.2: w0min = -1.2
    else:
        if w0max < -0.5: w0max = -0.5
        if w0min > -1.1: w0min = -1.1

    norm = mcolors.Normalize(vmin=w0min, vmax=w0max)

    fig = plt.figure(figsize=(10, 3.5))
    gs = fig.add_gridspec(
            1, 2,
        wspace=0.32, hspace=0.3
    )

    ax00 = fig.add_subplot(gs[0, 0])
    ax01 = fig.add_subplot(gs[0, 1])

    axes = {'00': ax00, '01': ax01}
    if AP_ISO:
        data = {'00': alph_ISO, '01': alph_AP}
        ylabels = {
            '00': r'$\alpha_\mathrm{ISO}(z)$',
            '01': r'$\alpha_\mathrm{AP}(z)$'
            }        
    else:
        data = {'00': alph_perp, '01': alph_par}
        ylabels = {
            '00': r'$\alpha_\perp(z)$',
            '01': r'$\alpha_\parallel(z)$'
            }


        # if envelope_settings is not None:
        #     envelope_type = envelope_settings['type']
        #     if envelope_type == 'spline':
        #         a = envelope_settings['alpha']
        #         k = envelope_settings['k']
        #         smooth = envelope_settings['smooth']
        #         from scipy.interpolate import UnivariateSpline
        #         z_fine = np.linspace(0, 3, 200)
        #         spline = UnivariateSpline(z_DESI_uncert, a_perp_err, s=smooth, k=k)  # s contrôle le lissage
        #         ax10.fill_between(z_fine, 1 - spline(z_fine), 1 + spline(z_fine), color='k', alpha=a)
        
        # from scipy.interpolate import interp1d

        # z_fine = np.linspace(min(z_DESI_uncert), max(z_DESI_uncert), 200)
        # z_fine = np.linspace(0, 3, 200)
        # err_interp = interp1d(z_DESI_uncert, a_perp_err, kind='cubic', fill_value='extrapolate')
        # ax10.fill_between(z_fine, 1 - err_interp(z_fine), 1 + err_interp(z_fine), color='k', alpha=0.2)
    
    if DESI_errors is not None:
        uncert, z_uncert = get_DESI_errors()
        a_perp_err = []
        a_par_err = []
        a_iso_err = []
        a_AP_err = []
        for idx, (_, value) in enumerate(uncert.items()):
            if idx == 0:
                a_iso_err.append(value['d_iso'])
            else:
                a_perp_err.append(value['d_perp'])
                a_par_err.append(value['d_par'])
                a_AP_err.append(value['d_AP'])
                a_iso_err.append(value['d_iso'])
        a_err = [a_iso_err, a_AP_err, a_perp_err, a_par_err]
    
    if AP_ISO:
        a_err = a_err[:2]
    else:
        a_err = a_err[2:]

    for idx, (key, ax) in enumerate(axes.items()):
        for i in range(len(w0)):
            ax.plot(redshift, data[key][i], color=cmap(norm(w0[i])))
    
        if DESI_errors is not None:
            if idx == 0 and AP_ISO:
                ax.errorbar(z_uncert, [1 for _ in z_uncert], yerr=a_err[idx], fmt='o', color='k', markersize=3)
            else:
                ax.errorbar(z_uncert[1:], [1 for _ in z_uncert[1:]], yerr=a_err[idx], fmt='o', color='k', markersize=3)

        ax.axhline(1, color='k', ls='--', lw=1)

        ax.tick_params(which='major', direction='in', length=4, width=1,   top=True, right=True)
        ax.tick_params(which='minor', direction='in', length=2, width=0.8, top=True, right=True)
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())

        ax.set_ylabel(ylabels[key], fontsize=18)
        ax.set_xlim(0, 3)

        if ylim is not None:
            if 'all' in ylim:
                ax.set_ylim(ylim['all'])
            elif key in ylim:
                ax.set_ylim(ylim[key])

        if yticks is not None:
            if 'all' in yticks:
                ax.yaxis.set_major_locator(MultipleLocator(yticks['all']))
            elif key in yticks:
                ax.yaxis.set_major_locator(MultipleLocator(yticks[key]))

        if xlabel and key in ('00', '01'):
            ax.set_xlabel('Redshift', fontsize=16)

    leg_txt = rf'$w_a = {slope:.2f}(w_0 + {offset:.1f})$'
    if slope == 0.0: leg_txt = rf'$w_a = 0$'
    text = AnchoredText(
        leg_txt,
        loc=text_loc, frameon=True, prop=dict(size=14)
    )
    axes['00'].add_artist(text)

    # --- Colorbar à droite, taille ~1 subplot, centrée ---
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])

    cbar_ax = fig.add_axes([0.92, 0.12, 0.015, 0.74])
    cb = fig.colorbar(sm, cax=cbar_ax)
    cb.set_label(r'$w_0$', fontsize=16)
    cb.ax.tick_params(labelsize=11)

    if path_save is not None:
        slope_str  = f'{slope:.2f}'
        offset_str = f'{offset:.1f}'
        
        if not xlabel:
            folder = r'\no_xlabel'
            pth_folder =  path_save + folder
            os.makedirs(pth_folder, exist_ok=True)
            fname = folder + rf'\{name}_{slope_str}_{offset_str}' if name else rf'{slope_str}_{offset_str}'
        else:
            fname = f'{name}_{slope_str}_{offset_str}' if name else f'{slope_str}_{offset_str}'

        if path_save is None:
            path_save = rf'..\..\figures\degen_line\alpha_ratio\final\{fname}.png'
        else:
            path_save= path_save + f'\{fname}.png'
        fig.savefig(
            path_save,
            dpi=150, bbox_inches='tight'
        )

    if show_plots:
        plt.show()
    else:
        plt.close(fig)

    return fig, axes

<string>:165: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:165: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<string>:165: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
<>:165: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
/tmp/ipykernel_55576/3596661773.py:165: SyntaxWarning: "\{" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\{"? A raw string is also an option.
  path_save= path_save + f'\{fname}.png'


In [9]:
z = np.linspace(0.0, 3.0, 100)
# a, b, c = alpha_perp_parallel(z, (-1.0, 0.0, 0.2975, 100))

w0 = np.linspace(-1.1, -0.5, 10)
offset = 1
slope = -3.66
wa = slope * ( w0 + offset)
Omega_m, hrdrag, _ = get_Om_hrdrag(w0, wa)
param_mirage = list(zip(w0, wa, Omega_m, hrdrag))

TypeError: only 0-dimensional arrays can be converted to Python scalars

In [23]:
# # plot_alpha_ratios(z, slope, offset, param_mirage, ylim={'all': (0.89, 1.1)}, yticks={'all': 0.05}, name='mirage')
# spline_settings = {'type': 'spline', 'alpha': 0.3, 'k': 2, 'smooth': 3}
# plot_alpha_ratios(z, slope, offset, param_mirage, ylim={'all': (0.89, 1.11)}, #yticks={'all': 0.05}, name='mirage',
#                     DESI_errors=get_DESI_errors(), envelope_settings=spline_settings)

In [24]:
z = np.linspace(0.0, 3.0, 100)
# a, b, c = alpha_perp_parallel(z, (-1.0, 0.0, 0.2975, 100))

w0 = np.linspace(-1.2, -0.8, 10)
offset = 1
slope = -3.66
wa = slope * ( w0 + offset)
Omega_m, hrdrag, _ = get_Om_hrdrag(w0, wa)
param_mirage_sym = list(zip(w0, wa, Omega_m, hrdrag))

In [25]:
# plot_alpha_ratios(z, slope, offset, param_mirage, ylim={'all': (0.89, 1.1)}, yticks={'all': 0.05}, name='mirage')
spline_settings = {'type': 'spline', 'alpha': 0.3, 'k': 2, 'smooth': 3}
# plot_alpha_ratios(z, slope, offset, param_mirage_sym, ylim={'all': (0.89, 1.11)}, #yticks={'all': 0.05}, name='mirage',
#                     DESI_errors=get_DESI_errors(), envelope_settings=spline_settings)

#### imitate Fig 13

In [26]:
# from matplotlib.offsetbox import AnchoredText

# alph_perp = []
# alph_par = []
# alph_AP = []
# alph_ISO = []

# z = np.linspace(0.0, 3.0, 1000)

# w0 = mean_w
# wa = mean_wa
# obh2 = mean_ombh2
# och2 = mean_omch2
# h = mean_H0/100
# hrdrag = mean_rdrag*h

# print(f"w0: {w0:.2f}, wa: {wa:.2f}, obh2: {obh2:.5f}, och2: {och2:.5f}, h: {h:.3f}, hrdrag: {hrdrag:.2f}")
# # Omegam = slopes[1] * w0[i] + intercepts[1]
# # hrdrag = slopes[2] * w0[i] + intercepts[2]
# a, b, c = alpha_perp_parallel_alt(z, (w0, wa, obh2, och2, h, hrdrag))
# alph_perp.append(a)
# alph_par.append(b)
# alph_AP.append(b/a)
# alph_ISO.append(c)

# w0 = -1.0
# wa = 0.0
# obh2 =  mean_ombh2_2
# och2 = mean_omch2_2
# h = mean_H0_2/100
# hrdrag = mean_rdrag_2*h
# a, b, c = alpha_perp_parallel_alt(z, (w0, wa, obh2, och2, h, hrdrag))
# alph_perp.append(a)
# alph_par.append(b)
# alph_AP.append(b/a)
# alph_ISO.append(c)


# # for DESY5 we only have omega_m and we'll set a value of hrdrag
# fid_planck = cosmoprimo.fiducial.Planck2018FullFlatLCDM()
# fid_bkg = fid_planck.get_background(engine='camb')
# fid_thermo = fid_planck.get_thermodynamics()
# rdrag_fid = fid_thermo.rs_drag
# DM_fid = fid_bkg.comoving_angular_distance(z)
# DH_fid = 1 / fid_bkg.efunc(z) * 2997.92458 # c/H(z) in Mpc, c=299792 km/s, H0 in km/s/Mpc
# DV_fid = (z * DM_fid**2 * DH_fid)**(1/3)
# DMover_rd_fid = DM_fid / rdrag_fid
# DHover_rd_fid = DH_fid / rdrag_fid
# DVover_rd_fid = DV_fid / rdrag_fid
# w0 = -1.0
# wa = 0.0
# omm =  mean_omm_3
# DM_desy5 = trans_comoving_dist_array(z, [omm, w0, wa])
# DH_desy5 = Hubble_dist(z, [omm, w0, wa])
# DV_desy5 = [compute_DV(red, [omm, w0, wa]) for red in z]
# rdrag_fid = 98.1
# DMover_rd_desy5 = [d / rdrag_fid for d in DM_desy5]
# DHover_rd_desy5 = [d / rdrag_fid for d in DH_desy5]
# DVover_rd_desy5 = [d / rdrag_fid for d in DV_desy5]
# alph_perp.append(DMover_rd_desy5 / DMover_rd_fid)
# alph_par.append(DHover_rd_desy5 / DHover_rd_fid)
# alph_ISO.append(DVover_rd_desy5 / DVover_rd_fid)